# Assignment 13: reversible LLM training (Colab runner)

This notebook runs the whole of assignment 13 on one Colab GPU (tested for a T4, 16 GB, fp16 autocast):

1. tokenize ~56M FineWeb-Edu tokens,
2. check that reversible back-prop gives the same gradients as ordinary autograd,
3. screen the reversible variants (midpoint, midpoint_a, leapfrog, hamiltonian) on short runs and pick one,
4. measure the maximum batch size, memory vs depth and throughput vs batch,
5. **run 1**: baseline GPT (20.1M params), 50M tokens, batch 64,
6. **run 2**: the chosen reversible variant, 50M tokens, same batch 64,
7. **run 3**: the chosen reversible variant, 50M tokens, at its maximum batch size,
8. build the results tables and plots (`experiments/assignment13/results/`).

All the logic is in the scripts under `experiments/assignment13/`. This notebook only calls them, so each step can also be run from a terminal, either one by one or all together with `python experiments/assignment13/run_all.py` (same stages, same defaults). See `experiments/assignment13/README.md` for the method and the findings.

Rough T4 timings: data 5-10 min, screening ~15 min, probes ~15-25 min, runs 1-3 about 10-20 min each.

## 0. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os
REPO_URL = "https://github.com/keshavamurali/chintana.git"
BRANCH = "assignment13"
REPO = "/content/chintana"

if not os.path.exists(REPO):
    !git clone -b {BRANCH} {REPO_URL} {REPO}
%cd {REPO}
!git log --oneline -1

# Colab already has a CUDA build of torch. Don't `pip install -e .`: pyproject.toml pins
# torch==2.2.2 for the author's Intel Mac and would replace it.
!pip -q install tiktoken omegaconf datasets

os.environ["PYTHONPATH"] = REPO
# the same allocator setting for every run and probe; reduces fragmentation at large batch
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
A13 = "experiments/assignment13"

In [ ]:
import torch
print(torch.__version__, torch.cuda.get_device_name(0))

## 1. Data: FineWeb-Edu, GPT-2 BPE

The script streams `HuggingFaceFW/fineweb-edu` (`sample-10BT`) and writes 1M validation tokens and 55M training tokens to `experiments/assignment13/data/fineweb_edu/`. Each run trains on 50M randomly sampled 512-token windows, which is a little under one epoch. Use `--source tinystories` if the Hub is slow.

In [ ]:
!python {A13}/prepare_data.py --source fineweb_edu --train-tokens 55000000 --val-tokens 1000000

## 2. Correctness: reversible gradients == autograd gradients

This checks every variant in float64 on the CPU, where the gradients must match to about 1e-15. It then repeats the check at full model size on the GPU under fp16 autocast. It also measures how far the rebuilt hidden states drift from the stored ones.

In [ ]:
!python {A13}/check_reversible.py

## 3. Variant screening (10M tokens each, batch 64)

The screening runs use the same init, data order and batch for every variant. The pick is the reversible variant with the lowest validation loss that did not diverge. Override `CHOSEN` by hand if you prefer.

In [ ]:
SCREEN_TOKENS = 10_000_000
for v in ["baseline", "midpoint", "midpoint_a", "leapfrog", "hamiltonian"]:
    !python {A13}/train.py rev.variant={v} run.name=screen_{v} data.batch_size=64 data.total_tokens={SCREEN_TOKENS} run.log_interval=50 run.n_evals=5

In [ ]:
import json, glob
rows = [json.load(open(p)) for p in sorted(glob.glob(f"{A13}/results/runs/screen_*/summary.json"))]
for r in rows:
    print(f"{r['variant']:12s} val {r['final_val_loss']:.4f}  train {r['final_train_loss']:.4f}  "
          f"{r['tokens_per_s']:>9,.0f} tok/s  peak {r.get('peak_mem_mb', 0)/1024:.2f} GB  diverged={r['diverged']}")
ok = [r for r in rows if r["variant"] != "baseline" and not r["diverged"]]
CHOSEN = min(ok, key=lambda r: r["final_val_loss"])["variant"]
print("\nCHOSEN =", CHOSEN)

## 4. Memory and throughput probes

This step finds the largest batch that survives a full training step, for the baseline and the chosen variant. Each is measured with both loss heads: `fused` is the chunked cross-entropy used in every run, and `standard` is the full logits tensor. It also measures tokens/s at powers of two up to each max, and peak memory vs depth at batch 32 (the paper's Fig. 3).

In [ ]:
!python {A13}/memory_probe.py --modes max,throughput,depth --variants baseline,{CHOSEN} --loss-impls fused,standard

In [ ]:
import csv
mb = {(r["variant"], r["loss_impl"]): int(r["max_batch"]) for r in csv.DictReader(open(f"{A13}/results/logs/max_batch.csv"))}
print(mb)
# keep ~5% headroom: the probe measures one step, and a long run fragments the allocator a little
RUN3_BATCH = max(64, int(mb[(CHOSEN, "fused")] * 0.95) // 8 * 8)
BASE_MAX_BATCH = max(8, int(mb[("baseline", "fused")] * 0.95) // 8 * 8)
print("run 3 batch:", RUN3_BATCH, "| baseline max (for the optional run 4):", BASE_MAX_BATCH)

## 5. Run 1: baseline, batch 64, 50M tokens

In [ ]:
!python {A13}/train.py {A13}/config/run1_baseline.yaml run.log_interval=50

## 6. Run 2: reversible (chosen variant), batch 64, 50M tokens

In [ ]:
!python {A13}/train.py {A13}/config/run2_reversible.yaml rev.variant={CHOSEN} run.log_interval=50

## 7. Run 3: reversible at its maximum batch, 50M tokens

The token budget is the same, so there are far fewer optimizer steps. The peak lr is scaled by sqrt(batch/64), capped at 3e-3.

In [ ]:
!python {A13}/train.py {A13}/config/run3_reversible_maxbatch.yaml rev.variant={CHOSEN} data.batch_size={RUN3_BATCH} run.log_interval=5

## 8. (Optional) Run 4: baseline at *its* maximum batch

This run isn't required. It separates two effects in run 3: what a bigger batch does, and what reversibility itself does.

In [ ]:
RUN_OPTIONAL = False
if RUN_OPTIONAL:
    !python {A13}/train.py {A13}/config/run1_baseline.yaml run.name=run4_baseline_maxbatch data.batch_size={BASE_MAX_BATCH} optim.lr_scaling=sqrt run.log_interval=10

## 9. Report

In [ ]:
!python {A13}/report.py

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open(f"{A13}/results/summary.md").read()))
for name in ["val_loss", "train_loss", "throughput", "peak_memory", "screen_val_loss",
             "memory_vs_depth", "memory_vs_batch", "throughput_vs_batch"]:
    path = f"{A13}/results/plots/{name}.png"
    if os.path.exists(path):
        display(Image(path))

## 10. Save the results

The results directory (CSVs, JSON, plots, `summary.md`) is small. Download the zip, or copy it to Drive, and commit it under `experiments/assignment13/results/`.

In [ ]:
!cd {A13} && zip -qr /content/assignment13_results.zip results
from google.colab import files
files.download("/content/assignment13_results.zip")